# Sector Click-to-Filter (overview → detail)

**Family:** Interactive, animated & big data &nbsp;|&nbsp; **Library:** Altair (Vega-Lite) &nbsp;|&nbsp; **Level:** Intermediate

> *Interactive, animated & big data:* What-if sliders, linked views, animation and million-point rendering. Use these when exploration or motion adds understanding that a static chart cannot.

## Design purpose

An overview bar chart of sector returns linked to a detail scatter of individual stocks. Clicking a sector highlights its stocks on the risk/return plane and fades the rest, following Shneiderman's mantra: overview first, zoom and filter, details on demand. Readers go from "which sector did well?" to "which stocks drove it, and at what risk?" in one gesture.

## When to use

- Drill down from an aggregate (sector, region, fund) to its members
- Compare one group's members against the whole universe without losing context
- Explore a screen or watchlist interactively

## Avoid when

- Readers who need every group at once (use small multiples)
- Fewer than about 20 members in total (one labelled scatter is enough)

## Alternatives

- Sector heat map (treemap)
- Risk-return scatter
- Small multiples by sector

## Design tips

- The PNG shows the default state (nothing selected); open the HTML and click a bar (shift-click adds more)
- Fade the unselected points to gray rather than hiding them, to keep the context
- Keep the scatter's axes fixed so the selection doesn't rescale the view
- Put names and numbers in tooltips, not labels, to keep the scatter readable

## Example

Simulated one-year returns and volatilities for 72 fictional stocks in eight sectors.

### Setup

In [1]:
import numpy as np
import pandas as pd
import altair as alt

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]


@alt.theme.register("vizlib", enable=True)
def vizlib_theme():
    return alt.theme.ThemeConfig({
        "config": {
            "background": "#fcfcfb",
            "view": {"stroke": None},
            "range": {"category": PALETTE},
            "font": "Helvetica, Arial, sans-serif",
            "title": {"anchor": "start", "fontSize": 14, "color": "#0b0b0b"},
            "axis": {"gridColor": "#e6e5e1", "domainColor": "#c9c8c3", "tickColor": "#c9c8c3",
                     "labelColor": "#52514e", "titleColor": "#52514e", "titleFontWeight": "normal"},
            "legend": {"labelColor": "#52514e", "titleColor": "#52514e"},
        }
    })

### Data

In [2]:
rng = np.random.default_rng(12)
sectors = {  # (mean one-year return, typical volatility)
    "Technology": (0.18, 0.34), "Industrials": (0.08, 0.24), "Financials": (0.09, 0.26),
    "Consumer": (0.05, 0.25), "Health care": (0.04, 0.22), "Utilities": (0.02, 0.15),
    "Materials": (-0.04, 0.28), "Energy": (-0.08, 0.33)}
rows = []
for sector, (mu, vol) in sectors.items():
    for i in range(9):
        rows.append({"ticker": f"{sector[:3].upper()}{i + 1}", "sector": sector,
                     "ret": round(rng.normal(mu, 0.09), 4), "vol": round(abs(rng.normal(vol, 0.045)), 4)})
stocks = pd.DataFrame(rows)
stocks.groupby("sector")[["ret", "vol"]].mean().round(3).sort_values("ret", ascending=False)

,ret,vol
sector,,
Technology,0.177,0.358
Financials,0.111,0.259
Industrials,0.081,0.215
Health care,0.065,0.214
Consumer,0.062,0.258
Utilities,0.016,0.139
Materials,-0.043,0.295
Energy,-0.061,0.321


### Plot

In [3]:
pick = alt.selection_point(fields=["sector"], name="pick")
order = stocks.groupby("sector")["ret"].mean().sort_values(ascending=False).index.tolist()

bars = (alt.Chart(stocks).mark_bar(cornerRadiusEnd=4, height=22)
        .encode(y=alt.Y("sector:N", sort=order, title=None),
                x=alt.X("mean(ret):Q", title="Average 1-year return", axis=alt.Axis(format="%")),
                color=alt.condition("datum.mean_ret >= 0", alt.value(PALETTE[0]), alt.value(PALETTE[1])),
                opacity=alt.condition(pick, alt.value(1), alt.value(0.3)),
                tooltip=["sector:N", alt.Tooltip("mean(ret):Q", title="Avg return", format=".1%")])
        .transform_joinaggregate(mean_ret="mean(ret)", groupby=["sector"])
        .add_params(pick)
        .properties(width=250, height=330, title=alt.Title("Click a sector", fontSize=12)))

scatter = (alt.Chart(stocks).mark_circle(size=90, stroke="#fcfcfb", strokeWidth=1)
           .encode(x=alt.X("vol:Q", title="Volatility (1-year)", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.5])),
                   y=alt.Y("ret:Q", title="1-year return", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[-0.35, 0.45])),
                   color=alt.condition(pick, alt.value(PALETTE[0]), alt.value("#d0cfca")),
                   opacity=alt.condition(pick, alt.value(0.95), alt.value(0.5)),
                   order=alt.condition(pick, alt.value(1), alt.value(0)),
                   tooltip=["ticker:N", "sector:N", alt.Tooltip("ret:Q", format="+.1%"), alt.Tooltip("vol:Q", format=".0%")])
           .properties(width=440, height=330, title=alt.Title("Stocks: risk vs. return", fontSize=12)))
zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#8f8e88").encode(y="y:Q")

chart = alt.hconcat(bars, scatter + zero, spacing=40).properties(
    title=alt.Title("From sector to stock: what drove each sector? (simulated)", subtitle=[
        "Click a bar to highlight its stocks; shift-click to add sectors; click empty space to reset.", " "]))
chart

alt.HConcatChart(...)

## How to read it

The bars rank sectors by average return: blue gained and orange lost. Click Technology and its stocks light up in the scatter, mostly high-return but also high-volatility. Click Utilities and they cluster at low risk and low return. Hover any dot for the stock's name and figures.

---
Standalone script: [`sector_click_filter.py`](sector_click_filter.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)